# Paper assistant (MCP client)

This notebook talks to the **paper-assistant MCP server** (`mcp_server/server.py`).
The server can (1) answer questions about one downloaded PDF using **RAG**, (2) summarize it, and
(3) run the existing multi-agent pipeline (`main.py`). The notebook only sends requests and shows the results.

**Before you start:** `pip install -r requirements.txt`, keys in `.env`, PDFs in `papers/`.
Run the cells from the top. Cells use top-level `await`.

In [1]:
# Setup: project paths and MCP client imports
import os
import sys
from pathlib import Path

import nbformat
from IPython.display import Markdown, display
from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SERVER = PROJECT / "mcp_server" / "server.py"
TIMEOUT_MIN = 30        # pipeline steps can take minutes
NOTES_NB = PROJECT / "notebooks" / "paper_notes.ipynb"     # where save_note() writes
SERVER_LOG = open(PROJECT / "notebooks" / "server.log", "a", encoding="utf-8")   # server log lines go here
print("server:", SERVER, "| exists:", SERVER.exists())

server: C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\mcp_server\server.py | exists: True


In [2]:
# The helper every cell uses: start the server (stdio), call ONE tool, return its text
def server_params():
    return StdioServerParameters(command=sys.executable, args=[str(SERVER)], env={**os.environ})


async def call(tool, **args):
    # errlog must be a real file: Jupyter's sys.stderr has no file handle (breaks on Windows)
    async with Client(stdio_client(server_params(), errlog=SERVER_LOG)) as client:
        res = await client.call_tool(tool, args, read_timeout_seconds=TIMEOUT_MIN * 60.0)
    text = "".join(getattr(c, "text", "") for c in res.content)
    if res.is_error:                      # a failing tool answers with is_error, it does not raise
        raise RuntimeError(text)
    return text


def show(text):
    display(Markdown(text))


def save_note(markdown_text, path=NOTES_NB):
    # append a markdown cell to a notebook file (created if missing)
    path = Path(path)
    nb = nbformat.read(path, as_version=4) if path.exists() else nbformat.v4.new_notebook()
    nb.cells.append(nbformat.v4.new_markdown_cell(markdown_text))
    nbformat.write(nb, path)
    print("saved to", path)

## 1. Which tools does the server offer?

In [3]:
# Ask the server what it can do (MCP discovery)
async def list_tools():
    async with Client(stdio_client(server_params(), errlog=SERVER_LOG)) as client:
        return (await client.list_tools()).tools


for t in await list_tools():
    print(f"- {t.name}: {(t.description or '').splitlines()[0]}")

- list_papers: List the PDFs found in the allowed folders (papers/ by default) and whether each is indexed.
- index_paper: Index one PDF (file name, part of the name, or full path) so it can be searched. Safe to repeat.
- index_folder: Index every PDF in a folder (default: the first allowed folder, normally papers/). Folder must be allowed.
- search_papers: Semantic search over indexed papers. Give `paper` to search only one PDF. Returns top chunks with page.
- ask_paper: Answer a specific question about ONE downloaded paper: retrieves the best passages (RAG), then the LLM
- summarize_paper: Summarize ONE paper (problem, method, results, limitations) from retrieved passages, with page citations.
- get_paper_sheet: Read the detailed sheet the pipeline's extractor already made (out/extracted_v3). With no `paper`, list the
- run_step: Run one stage of the paper pipeline (main.py --step ...). step: extract|analyze|literature|write|review|fix|
- extract_sheets: Stage 1: read every PDF in pa

## 2. Your papers
`list_papers` shows the PDFs in `papers/`. `index_folder` reads them into the RAG index
(only needed once per file; unchanged files are skipped).

In [4]:
show(await call("list_papers"))

- 1706.03762v7.pdf (2.2 MB)  C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\papers\1706.03762v7.pdf
- 2201.11903v6.pdf (0.9 MB)  C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\papers\2201.11903v6.pdf
- 2210.03629v3.pdf (0.6 MB)  C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\papers\2210.03629v3.pdf
- 2302.04761v1.pdf (0.7 MB)  C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\papers\2302.04761v1.pdf
- 2304.03442v2.pdf (11.9 MB)  C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\papers\2304.03442v2.pdf

In [5]:
print(await call("index_folder"))

[
 {
  "name": "1706.03762v7.pdf",
  "status": "indexed",
  "pages": 15,
  "chunks": 49,
  "pages_without_text": 0
 },
 {
  "name": "2201.11903v6.pdf",
  "status": "indexed",
  "pages": 43,
  "chunks": 174,
  "pages_without_text": 0
 },
 {
  "name": "2210.03629v3.pdf",
  "status": "indexed",
  "pages": 33,
  "chunks": 138,
  "pages_without_text": 0
 },
 {
  "name": "2302.04761v1.pdf",
  "status": "indexed",
  "pages": 17,
  "chunks": 89,
  "pages_without_text": 0
 },
 {
  "name": "2304.03442v2.pdf",
  "status": "indexed",
  "pages": 22,
  "chunks": 159,
  "pages_without_text": 0
 }
]


## 3. Ask a specific question about ONE paper (RAG)
Write the PDF name (or part of it) and your question. The server finds the best passages, and the LLM answers
**only from them**, with page numbers.

In [6]:
PAPER = "1706.03762"          # file name or part of it
QUESTION = "How many BLEU points does the big model get on English-to-German, and how long was it trained?"

answer = await call("ask_paper", pdf=PAPER, question=QUESTION, k=6)
show(answer)

The big model achieves a BLEU score of 28.4 on the English-to-German translation task and was trained for 3.5 days.

**Supporting details:**
*   "On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big) in Table 2) ... establishing a new state-of-the-art BLEU score of 28.4." (p. 8)
*   "Training took 3.5 days on 8 P100 GPUs." (p. 8)

_Sources: 1706.03762v7.pdf, pages 7, 8, 9_

In [7]:
# Write the answer into a notebook as a markdown cell
save_note(f"### {QUESTION}\n\n{answer}")

saved to C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\notebooks\paper_notes.ipynb


## 4. Summarize one paper
Retrieves passages for *problem, method, results, limitations*, then writes a cited summary.

In [8]:
summary = await call("summarize_paper", pdf=PAPER)
show(summary)

# 1706.03762v7.pdf

## Problem
Learning long-range dependencies is a key challenge in sequence transduction tasks (p. 6). Previous models often relied on recurrent or convolutional layers, which the authors aimed to replace with a self-attention mechanism to improve computational efficiency and parallelization (p. 6).

## Method
The Transformer architecture replaces recurrent layers with stacked self-attention and point-wise, fully connected layers for both the encoder and decoder (p. 3). The encoder and decoder each consist of a stack of $N=6$ identical layers (p. 3). Each layer includes a multi-head self-attention mechanism and a position-wise feed-forward network, with residual connections and layer normalization applied to each sub-layer (p. 3). The model uses a parameter-free position representation and scaled dot-product attention (p. 1).

## Results
The Transformer was evaluated on the WMT 2014 English-to-German and English-to-French translation tasks (p. 7). The "big" Transformer model achieved a new state-of-the-art BLEU score of 28.4 on the English-to-German task, outperforming previous models by more than 2.0 BLEU (p. 8). On the English-to-French task, the big model achieved a BLEU score of 41.0 (p. 8). These results were attained at a fraction of the training cost of competitive models (p. 8).

## Limitations
The authors noted that reducing the attention key size $d_k$ hurts model quality, suggesting that a more sophisticated compatibility function than the dot product might be beneficial (p. 9). Additionally, while bigger models generally perform better, quality drops off if there are too many attention heads (p. 9). The model also relies on label smoothing, which improves accuracy and BLEU scores but hurts perplexity because the model learns to be more unsure (p. 8).

## One-sentence takeaway
The Transformer is a novel architecture that replaces recurrent layers with self-attention, achieving state-of-the-art translation performance with significantly reduced training costs.

_Sources: pages 1, 3, 6, 7, 8, 9, 11, 12, 13, 14, 15_

In [10]:
save_note(summary)

saved to C:\Users\inder\Projects\Scientific-Paper-Data-Extractor\notebooks\paper_notes.ipynb


## 5. Search across all indexed papers
No LLM involved: just the most relevant passages, with file and page.

In [11]:
show(await call("search_papers", query="how an agent stores and retrieves memories", k=4))

**2304.03442v2.pdf p. 8** (distance 0.2956)
UIST ’23, October 29-November 1, 2023, San Francisco, CA, USA J.S. Park, J.C. O’Brien, C.J. Cai, M.R. Morris, P. Liang, M.S. Bernstein Figure 5: Our generative agent architecture. Agents perceive their environment, and all perceptions are saved in a comprehensive record of the agent’s experiences called the memory stream. Based on their perceptions, the architecture retrieves relevant memories and uses those retrieved actions to determine an action. These retrieved memories are also used to form longer-term plans and create higher-level reflections, both of which are entered into the memory stream for future use. 4 GENERATIVE AGENT ARCHITECTURE Generative agents aim to provide a framework for behavior in an open world: one that can engage in interactions with other agents and react to changes in the environment. Generative agents take their current environment and past experiences as input and generate behavior as output. Underlying this behavior is a novel agent architecture that

**2304.03442v2.pdf p. 21** (distance 0.3522)
that, I usually go to the library to work on my research paper, taking breaks for lunch at Hobbs Cafe and a walk in the park. In the evening, I often return to the library until around 6:00 pm. B.2 Memory The questions on agents’ memory probe the agents’ ability to remember their past experiences and retrieve the relevant pieces of memory. The names in the square brackets of the first and last questions are randomly chosen from the names of other agents with whom the subject agent has interacted. • Who is [Wolfgang Schulz]? Wolfgang Schulz is a person I met recently who has been helping me with my research paper on the effects of gentrification on low-income communities. We had several conversations about it, he gave me some great resources, and he even invited me to join him and another friend for lunch. He seems really curious about the issue and has a good grasp on it too. I found that really interesting. • Who is Kane Martinez? I’m sorry, I’m not sure who Kane Martinez is. Could

**2304.03442v2.pdf p. 8** (distance 0.3604)
the memory stream described below surfaces relevant memories, resulting in a more informative and specific response that mentions Isabella’s passion for making people feel welcome and included, planning events and creating an atmosphere that people can enjoy, such as the Valentine’s Day party. Approach: The memory stream maintains a comprehensive record of the agent’s experience. It is a list of memory objects, where each object contains a natural language description, a creation timestamp, and a most recent access timestamp. The most basic element of the memory stream is an observation, which is an event directly perceived by an agent. Common observations include behaviors performed by the agent themselves or behaviors that agents perceive being performed by other agents or non-agent objects. For instance, Isabella Rodriguez, who works at a coffee shop, might accrue the following observations over time: (1)Isabella Rodriguez is setting out the pastries, (2) Maria Lopez is studying

**2304.03442v2.pdf p. 13** (distance 0.368)
hypothetical situations for which the agent needs to respond believably: “Your breakfast is burning! What would you do?” • Reflections: We ask questions that require the agents to leverage their deeper understanding of others and themselves gained through higher-level inferences, such as “If you were to spend time with one person you met recently, who would it be and why?” The full list of questions and a sample of agent responses are included in Appendix B. Agents were sampled from the end of a two game day simulation with the full architecture, during which they had accumulated a number of interactions and memories that would shape their responses. To gather feedback on the believability of the responses, we recruited participants as human evaluators and tasked them with watching a replay of a randomly chosen agent’s life in Smallville. Participants had access to all information stored in the agent’s memory stream. The study followed a within-subjects design, where 100 participants

## 6. Use what the pipeline already extracted
The extractor already built a detailed sheet per paper (`out/extracted_v3`).

In [12]:
print(await call("get_paper_sheet"))                    # list the sheets
print(await call("get_paper_sheet", paper="react", fields="title,limitations"))

- [Vaswani et al., 2017] Attention Is All You Need (1706.03762v7.pdf)
- [Wei et al., 2022] Chain-of-Thought Prompting Elicits Reasoning in Large Language Models (2201.11903v6.pdf)
- [Yao et al., 2023] ReAct: Synergizing Reasoning and Acting in Language Models (2210.03629v3.pdf)
- [Schick et al., 2023] Toolformer: Language Models Can Teach Themselves to Use Tools (2302.04761v1.pdf)
- [Park et al., 2023] Generative Agents: Interactive Simulacra of Human Behavior (2304.03442v2.pdf)
{
 "title": "ReAct: Synergizing Reasoning and Acting in Language Models",
 "limitations": [
  "Complex tasks with large action spaces require more demonstrations to learn well, which can exceed the input length limit of in-context learning.",
  "Structural constraints of interleaving reasoning and acting can reduce flexibility in formulating reasoning steps, leading to reasoning errors.",
  "Non-informative search results can derail model reasoning.",
  "The model can get stuck in repetitive loops of thoughts a

## 7. Run the multi-agent pipeline, stage by stage
Each stage calls `python main.py --step ...` inside the server. They use the LLM and take minutes.

In [13]:
show(await call("extract_sheets"))

exit code: 0

```
[1] skipping 1706.03762v7.pdf (already extracted)
[1] skipping 2201.11903v6.pdf (already extracted)
[1] skipping 2210.03629v3.pdf (already extracted)
[1] skipping 2302.04761v1.pdf (already extracted)
[1] skipping 2304.03442v2.pdf (already extracted)
[todo] missing related work: Reflexion: Language Agents with Verbal Reinforcement Learning
[todo] missing related work: Self-Refine: Iterative Refinement with Self-Feedback
[todo] missing related work: Voyager: An Open-Ended Embodied Agent with Large Language Models
Done. See the out/ folder; human checklist: README.md, out/human_todo.md
```

In [ ]:
TOPIC = "Memory and tool use for LLM agents"
show(await call("analyze_topic", topic=TOPIC))

In [15]:
show(await call("write_draft", topic=TOPIC, venue="Conference"))

In [16]:
show(await call("review_draft"))

exit code: 0

```
[4] reviewing the draft (claims vs evidence)
  6 claims; 0 UNSUPPORTED; 1 abstract/conclusion mismatches
[lint] out\lint_report.md: no issues
[todo] missing related work: LangChain
[todo] missing related work: AutoGPT
[todo] missing related work: BabyAGI
Done. See the out/ folder; human checklist: README.md, out/human_todo.md
```

## 8. Read the results

In [17]:
print(await call("list_outputs"))

- analysis.json (10445 bytes)
- change_log.md (252 bytes)
- claims_vs_evidence.md (1581 bytes)
- extracted_v3/1706.03762v7.json (14551 bytes)
- extracted_v3/2201.11903v6.json (17211 bytes)
- extracted_v3/2210.03629v3.json (15413 bytes)
- extracted_v3/2302.04761v1.json (14510 bytes)
- extracted_v3/2304.03442v2.json (16780 bytes)
- human_todo.md (396 bytes)
- input_hash.json (76 bytes)
- latex_build.log (46 bytes)
- lint_report.md (440 bytes)
- paper.tex (29683 bytes)
- paper_draft.md (33225 bytes)
- reading_list.md (673 bytes)
- refs.bib (3268 bytes)
- review.json (2947 bytes)
- sections/00_abstract.md (1705 bytes)
- sections/01_introduction.md (5124 bytes)
- sections/02_related-work.md (6160 bytes)
- sections/03_method.md (7728 bytes)
- sections/04_proposed-evaluation.md (4667 bytes)
- sections/05_discussion-and-limitations.md (3453 bytes)
- sections/05_discussion.md (3445 bytes)
- sections/06_limitations.md (2401 bytes)
- sections/07_conclusion.md (749 bytes)


In [18]:
draft = await call("read_output", name="paper_draft.md", max_chars=6000)
show(draft)

# Cognitive Architectures for Autonomous Agents: A Unified Framework for Memory and Tool-Use

## Abstract

Large language models have demonstrated significant potential in linguistic synthesis and logical reasoning; however, existing agentic frameworks often struggle to maintain long-term coherence while grounding reasoning in external reality. Current paradigms frequently treat reasoning and acting as isolated components, leading to structural limitations in recursive tool-chaining and persistent memory synthesis. To address these gaps, we propose **Cognitive Orchestration**, a unified framework that formalizes agentic behavior as a recursive, tuple-based memory update and retrieval process. We define the agent's internal state through a formal memory object, $M = (t, c, a, o, r)$, where $t$ represents the timestamp, $c$ the context, $a$ the action, $o$ the observation, and $r$ the reflection. This structure enables the synthesis of past experiences into insights that guide future tool selection. By integrating a differentiable retrieval score, $S(q, m) = w_1 \cdot \text{sim}(q, c) + w_2 \cdot \exp(-\lambda \cdot (t_{\text{now}} - t_m))$, the framework balances semantic relevance with temporal recency to improve grounded task performance. We hypothesize that this unified approach will enhance multi-step arithmetic reasoning, long-term task coherence, and factual grounding. This paper details the theoretical design of the Cognitive Orchestration architecture and outlines a proposed evaluation plan using benchmarks such as GSM8K, ALFWorld, and ASDiv to test our hypotheses regarding accuracy, TrueSkill rating, and hallucination reduction. We aim to establish a standard for autonomous agents capable of sustained, multi-step reasoning in dynamic, open-world environments.

## Introduction

Large language models (LLMs) have emerged as powerful engines for linguistic synthesis and logical reasoning [Vaswani et al., 2017]. Recent advancements, such as Chain-of-Thought (CoT) prompting, have demonstrated that providing models with intermediate reasoning steps can improve performance on complex tasks like arithmetic and multi-hop inference [Wei et al., 2022]. However, these reasoning traces often function as static sequences that lack grounding in external reality. While paradigms like ReAct attempt to bridge this divide by interleaving reasoning with external tool usage, they often encounter structural constraints that limit the flexibility of the agent's planning process [Yao et al., 2023]. Furthermore, existing frameworks for autonomous agents frequently face challenges in balancing immediate reasoning accuracy with the long-term coherence required for persistent, multi-step tasks [Park et al., 2023].

Current agentic frameworks face a critical research gap: the lack of a unified mechanism for recursive tool-chaining and long-term memory synthesis. While models like Toolformer have shown that LLMs can learn to use tools in a self-supervised manner, they remain limited by their inability to chain tool outputs, where the result of one action serves as the input for a subsequent operation [Schick et al., 2023]. Similarly, while generative agents have successfully utilized memory and reflection to maintain social dynamics, their retrieval processes are often decoupled from the immediate requirements of tool-based problem solving [Park et al., 2023]. Consequently, agents often struggle to maintain logical consistency across extended interaction sequences, which we hypothesize leads to error propagation and hallucination in grounded tasks.

To address these limitations, we propose **Cognitive Orchestration**, a unified framework that formalizes agentic behavior as a recursive, tuple-based memory update and retrieval process. Cognitive Orchestration moves beyond simple prompting by defining the agent’s internal state through a formal memory object, $M = (t, c, a, o, r)$, where $t$ represents the timestamp, $c$ denotes the context, $a$ signifies the action, $o$ stands for the observation, and $r$ encapsulates the reflection. This structure allows the agent to synthesize past experiences into higher-level insights that guide future tool selection and execution. By formalizing the memory-reflection loop, we hypothesize that our framework enables the agent to perform multi-step tool chaining, where the reflection $r$ at time $t$ informs the context $c$ for the next recursive step, thereby addressing the structural limitations inherent in previous interleaving approaches [Yao et al., 2023].

Our framework introduces three primary contributions to the field of autonomous agentic reasoning. First, we define a memory tuple that serves as the foundation for both short-term reasoning and long-term behavioral coherence. Second, we introduce a recursive tool-chaining mechanism that allows agents to decompose complex goals into sub-tasks, execute them sequentially, and update their internal reflection trace based on the resulting observations. This mechanism is designed to address the inability of current models to chain tool outputs [Schick et al., 2023]. Third, we propose a retrieval score, $S(q, m) = w_1 \cdot \text{sim}(q, c) + w_2 \cdot \exp(-\lambda \cdot (t_{\text{now}} - t_m))$, which balances the semantic relevance of a memory $m$ to a query $q$ with its temporal recency. We hypothesize that by integrating this score into the retrieval process, the agent can reduce hallucination in grounded tasks by prioritizing contextually relevant observations over outdated information.

We position Cognitive Orchestration as a comprehensive architecture that synthesizes existing themes in agentic research—specifically the synergy between reasoning and acting [Wei et al., 2022], self-supervised tool integration [Schick et al., 2023], and long-term cognitive architectures [Park et al., 2023]. Unlike previous approaches that treat these components as distinct modules, our framework integrates them into a single loop where

[... cut, 27039 more characters]

### Writing into *this* notebook
`save_note()` writes to `notebooks/paper_notes.ipynb`. To write into this notebook instead, set
`NOTES_NB = PROJECT / "notebooks" / "paper_assistant.ipynb"` and then use **File > Reload Notebook from Disk**
(Jupyter does not refresh a notebook that was changed on disk behind its back).